# Explore Raw Data

Exploratory analysis of the bronze layer directly from PostgreSQL. This notebook examines both raw signal data and piece identification, then correlates them to understand the dataset before cleaning.

### What this notebook covers

1. **Raw signals** — record counts, signal types, value distributions, sampling frequency
2. **Combined view** — pieces with die matrix, cumulative travel times, and data quality
3. **Per die matrix** — travel time statistics, comparisons, variability
4. **Production patterns** — daily volumes, die matrix usage over time

### Column reference

All lifetime values are **cumulative piece travel times in seconds** from furnace exit.

| Signal | Process stage | Typical |
|---|---|---|
| `lifetime_first` | Main press — 2nd strike (1st op) | ~17–19s |
| `lifetime_second` | Main press — 3rd strike (2nd op) | ~24–26s |
| `lifetime_drill` | Main press — 4th strike (drill) | ~37–40s |
| `lifetime_auxiliary_press` | Auxiliary press | ~54–57s |
| `lifetime_bath` | Quench bath | ~56–59s |
| `lifetime` | General (≈ bath) | ~56–59s |
| `upsetting_lifetime` | Main press — 1st strike (upsetting) | ⚠️ Bad data, discard |

**Important**: These are per-piece travel times (~58s total), NOT OEE cycle time (11–16s between consecutive pieces).

In [ ]:
# TODO: implement this cell


---
## Part 1: Raw Signal Exploration

### Dataset overview

In [ ]:
# TODO: implement this cell


### Sample data

In [ ]:
# TODO: implement this cell


### Records per signal

In [ ]:
# TODO: implement this cell


### Value statistics per signal

All values are cumulative piece travel times in seconds from furnace exit.

In [ ]:
# TODO: implement this cell


### Value distribution per signal

Percentile distribution and zero-value count. Zero values indicate tracking failures (~1.2% for most signals, 22.8% for upsetting — confirming it is bad data).

In [ ]:
# TODO: implement this cell


### Sampling frequency

Time interval between consecutive readings for the same signal. The median ~14s reflects the production cadence (one piece every ~14 seconds). Large max gaps (up to 353 hours) correspond to weekends or machine stops.

In [ ]:
# TODO: implement this cell


---
## Part 2: Combined Piece View

Join lifetime signals with piece identification (piece_id, die_matrix) using the `bronze.v_pieces` view. This gives one row per piece with all cumulative times.

In [ ]:
# TODO: implement this cell


### Records per die matrix

How many records and unique pieces each die matrix has, and the time period it was active. Most production days show a single active matrix, but changeovers can happen mid-day.

In [ ]:
# TODO: implement this cell


---
## Part 3: Per Die Matrix Analysis

### Piece travel time statistics per die matrix

Each value is the elapsed time in seconds of a single piece traveling from furnace exit to a given process stage. These are NOT production cycle times (11–16s).

In [ ]:
# TODO: implement this cell


### Travel time comparison across die matrices

Side-by-side median piece travel time (seconds) for each stage. Differences between matrices reflect die-specific tooling geometry and process parameters.

In [ ]:
# TODO: implement this cell


### Cumulative travel time profile per die matrix

The process order: **Furnace → 2nd strike (~18s) → 3rd strike (~25s) → 4th strike (~38s) → Aux. press (~55s) → Quench bath (~58s)**

Values must be monotonically increasing. Non-increasing values indicate measurement errors.

In [ ]:
# TODO: implement this cell


### Time spent between stages (per die matrix)

Computed by subtracting consecutive cumulative times. These partial times identify which segment causes delays.

| Partial | Calculation | What happens |
|---|---|---|
| Furnace → 2nd strike | `lifetime_2nd_strike_s` | Robot pick, transfer, positioning at main press |
| 2nd strike → 3rd strike | `lifetime_3rd - lifetime_2nd` | Press retraction, repositioning |
| 3rd strike → 4th strike | `lifetime_4th - lifetime_3rd` | Transfer to drill station |
| 4th strike → Aux. press | `lifetime_aux - lifetime_4th` | Exit main press, transfer to auxiliary press, deburring and coining |
| Aux. press → Bath | `lifetime_bath - lifetime_aux` | Transport from auxiliary press to quench bath |

In [ ]:
# TODO: implement this cell


### Zero values and anomalies per die matrix

- **Zeros**: tracking failures (value = 0.00s). Should be removed during cleaning.
- **Outliers (3×IQR)**: extreme values from stuck pieces, unclosed cycles, or machine stops.

In [ ]:
# TODO: implement this cell


---
## Part 4: Production Patterns

### Daily production per die matrix

Number of pieces processed per day. Shows production volume, die matrix usage over time, and days with low counts (partial shifts, changeovers, maintenance).

In [ ]:
# TODO: implement this cell


### Daily record count per signal

In [ ]:
# TODO: implement this cell
